# Tree Builder
Builds a **3-level document tree**: Root → Chapters → Sections  
Stores the result in the `Tree` table as `treeJson`.  

**Run order:** `pageMetadata.ipynb` must be run first so every Page has `metadata`.

In [99]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. Imports & Connection

In [100]:
import json

from src.config.db import get_connection
from src.config.llm import llm

conn = get_connection()


## 2. Set Document ID
Change `document_id` to the document you want to build a tree for.

In [101]:
document_id = "DOC000001"


## 3. Fetch Pages with Metadata
Pulls every page for the document including the `metadata` JSON column.  
If a page has no metadata, its title/summary will be empty strings.

In [102]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            id,
            "pageNumber",
            content,
            metadata
        FROM "Page"
        WHERE "documentId" = %s
        ORDER BY "pageNumber"
        """,
        (document_id,),
    )
    rows = cur.fetchall()

pages = []
for row in rows:
    meta = row[3] if row[3] else {}
    pages.append({
        "id":         row[0],
        "pageNumber": row[1],
        "content":    row[2],
        "title":      meta.get("title", ""),
        "summary":    meta.get("summary", ""),
        "keywords":   meta.get("keywords", []),
        "topics":     meta.get("topics", []),
        "pageType":   meta.get("pageType", "content"),
    })

print(f"Fetched {len(pages)} pages")
pages[:3]


Fetched 50 pages


[{'id': 'DOC000001_P001',
  'pageNumber': 1,
  'content': 'REPORT TO CONGRESS\n110th\nAnnual Repor t of the Board of\nGovernors of the F ederal Reser ve System\n2023\nBOARD OF GO VERNORS OF THE FEDERAL RESER VE SYSTEM',
  'title': 'REPORT TO CONGRESS',
  'summary': '110th Annual Report of the Board of Governors of the Federal Reserve System',
  'keywords': ['Federal Reserve', 'Annual Report', 'Board of Governors'],
  'topics': ['Economy', 'Monetary Policy', 'Financial Stability'],
  'pageType': 'cover'},
 {'id': 'DOC000001_P002',
  'pageNumber': 2,
  'content': '',
  'title': '',
  'summary': '',
  'keywords': [],
  'topics': [],
  'pageType': 'content'},
 {'id': 'DOC000001_P003',
  'pageNumber': 3,
  'content': 'Contents\nAbout the F ederal Reser ve........................................................................................... iii\n1Overview....................................................................................................................... 1\n2Monetar y 

## 4. LLM Prompt — Build Tree Structure
We send page summaries + titles to the LLM and ask it to group them into  
**Chapters** (level 1) and **Sections** (level 2) under a single **Root** node.  

The LLM returns **only JSON** — no prose.

In [103]:
TREE_PROMPT = """
You are building a 3-level hierarchical tree for a document in a Vectorless RAG system.

Tree structure:
  root     (level 0) — the whole document
  chapter  (level 1) — major topic groups
  section  (level 2) — leaf nodes covering one or more consecutive pages

Rules:
- Every page must appear in exactly one section.
- A section spans one or more consecutive pages on the same sub-topic.
- A chapter groups related sections.
- Chapter and section titles must be descriptive (not "Chapter 1").
- pageIds must be the exact id strings provided — do not invent new ones.

Return ONLY valid JSON. No markdown, no explanation.

Schema:
{{
  "title": "<document root title>",
  "type": "root",
  "children": [
    {{
      "title": "<chapter title>",
      "type": "chapter",
      "children": [
        {{
          "title": "<section title>",
          "type": "section",
          "pageStart": <int>,
          "pageEnd":   <int>,
          "pageIds":   ["<page_id>", ...]
        }}
      ]
    }}
  ]
}}

Pages (pageNumber | page_id | title | summary):
{pages}
"""


## 5. Format Page List for Prompt

In [104]:
def format_pages_for_prompt(pages):
    lines = []
    for p in pages:
        summary_short = (p["summary"] or "")[:200].replace("\n", " ")
        lines.append(
            f"Page {p['pageNumber']} | {p['id']} | {p['title'] or '(no title)'} | {summary_short}"
        )
    return "\n".join(lines)

pages_text = format_pages_for_prompt(pages)
print(pages_text[:1000])


Page 1 | DOC000001_P001 | REPORT TO CONGRESS | 110th Annual Report of the Board of Governors of the Federal Reserve System
Page 2 | DOC000001_P002 | (no title) | 
Page 3 | DOC000001_P003 | Contents | Table of contents for the 110th Annual Report of the Board of Governors of the Federal Reserve System
Page 4 | DOC000001_P004 | Appendixes | Appendixes for the 110th Annual Report of the Board of Governors of the Federal Reserve System
Page 5 | DOC000001_P005 | About the Federal Reserve | Introduction to the Federal Reserve and its history
Page 6 | DOC000001_P006 | (no title) | 
Page 7 | DOC000001_P007 | Overview | This report covers the calendar-year 2023 operations and activities of the Federal Reserve, the central bank of the United States.
Page 8 | DOC000001_P008 | Additional Information | Additional information for calendar-year 2023 on Federal Reserve leadership, policy actions, budgets, and historical data can be found in the appendixes.
Page 9 | DOC000001_P009 | Monetary Policy and

## 6. Call LLM to Generate Tree

In [105]:
def call_llm_for_tree(pages_text):
    prompt = TREE_PROMPT.format(pages=pages_text)
    response = llm.invoke(prompt)
    text = response.content.strip()

    # Strip markdown code fences if present
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    return json.loads(text.strip())


# Fetch document title for the root node label
with conn.cursor() as cur:
    cur.execute(
        'SELECT title FROM "Document" WHERE id = %s',
        (document_id,),
    )
    doc_row = cur.fetchone()
doc_title = doc_row[0] if doc_row else "Document"

raw_tree = call_llm_for_tree(pages_text)

# Override root title with actual document title
raw_tree["title"] = doc_title

print(json.dumps(raw_tree, indent=2)[:2000])


{
  "title": "2023-annual-report-truncated",
  "type": "root",
  "children": [
    {
      "title": "Introduction",
      "type": "chapter",
      "children": [
        {
          "title": "Contents",
          "type": "section",
          "pageStart": 3,
          "pageEnd": 4,
          "pageIds": [
            "DOC000001_P003"
          ]
        },
        {
          "title": "About the Federal Reserve",
          "type": "section",
          "pageStart": 5,
          "pageEnd": 6,
          "pageIds": [
            "DOC000001_P005"
          ]
        },
        {
          "title": "Overview",
          "type": "section",
          "pageStart": 7,
          "pageEnd": 8,
          "pageIds": [
            "DOC000001_P007"
          ]
        },
        {
          "title": "Additional Information",
          "type": "section",
          "pageStart": 9,
          "pageEnd": 10,
          "pageIds": [
            "DOC000001_P008"
          ]
        },
        {
          "title"

## 7. Validate & Enrich Tree
- Checks every `pageId` from the DB is assigned.
- Derives `pageStart`, `pageEnd`, `pageCount` for chapters from their sections.
- Assigns stable `path` strings (`root`, `1`, `1.1`, `1.2`, …) for traversal.
- Warns about missing pages.

In [106]:
def enrich_and_validate(tree: dict, all_pages: list) -> dict:
    all_page_ids   = {p["id"] for p in all_pages}
    page_id_to_num = {p["id"]: p["pageNumber"] for p in all_pages}
    seen_ids: set  = set()

    chapter_idx = 0
    for chapter in tree.get("children", []):
        chapter_idx += 1
        chapter["level"] = 1
        chapter["path"]  = str(chapter_idx)

        section_idx        = 0
        chapter_page_count = 0
        chapter_page_start = None
        chapter_page_end   = None

        for section in chapter.get("children", []):
            section_idx += 1
            section["level"] = 2
            section["path"]  = f"{chapter_idx}.{section_idx}"

            # Re-derive page range from pageIds for consistency
            s_ids  = section.get("pageIds", [])
            s_nums = sorted(
                [page_id_to_num[pid] for pid in s_ids if pid in page_id_to_num]
            )

            if s_nums:
                section["pageStart"] = s_nums[0]
                section["pageEnd"]   = s_nums[-1]
            section["pageCount"] = len(s_ids)

            seen_ids.update(s_ids)
            chapter_page_count += len(s_ids)

            if s_nums:
                if chapter_page_start is None or s_nums[0] < chapter_page_start:
                    chapter_page_start = s_nums[0]
                if chapter_page_end is None or s_nums[-1] > chapter_page_end:
                    chapter_page_end = s_nums[-1]

        chapter["pageStart"] = chapter_page_start
        chapter["pageEnd"]   = chapter_page_end
        chapter["pageCount"] = chapter_page_count

    # Root stats
    tree["level"]     = 0
    tree["path"]      = "root"
    tree["pageCount"] = len(all_pages)
    tree["pageStart"] = min(p["pageNumber"] for p in all_pages)
    tree["pageEnd"]   = max(p["pageNumber"] for p in all_pages)

    # Missing page check
    missing = all_page_ids - seen_ids
    if missing:
        missing_nums = sorted(
            [page_id_to_num[pid] for pid in missing if pid in page_id_to_num]
        )
        print(f"WARNING: {len(missing)} pages not assigned to any section: pages {missing_nums}")
    else:
        print("✓ All pages accounted for.")

    return tree


enriched_tree = enrich_and_validate(raw_tree, pages)
print(json.dumps(enriched_tree, indent=2)[:2000])


{
  "title": "2023-annual-report-truncated",
  "type": "root",
  "children": [
    {
      "title": "Introduction",
      "type": "chapter",
      "children": [
        {
          "title": "Contents",
          "type": "section",
          "pageStart": 3,
          "pageEnd": 3,
          "pageIds": [
            "DOC000001_P003"
          ],
          "level": 2,
          "path": "1.1",
          "pageCount": 1
        },
        {
          "title": "About the Federal Reserve",
          "type": "section",
          "pageStart": 5,
          "pageEnd": 5,
          "pageIds": [
            "DOC000001_P005"
          ],
          "level": 2,
          "path": "1.2",
          "pageCount": 1
        },
        {
          "title": "Overview",
          "type": "section",
          "pageStart": 7,
          "pageEnd": 7,
          "pageIds": [
            "DOC000001_P007"
          ],
          "level": 2,
          "path": "1.3",
          "pageCount": 1
        },
        {
        

## 8. In-Memory Tree Nodes & Traversal Helpers
These helpers let you query the tree by page number or section path.

In [ ]:
class TreeNode:
    """Lightweight wrapper around a tree dict node."""

    def __init__(self, data: dict, parent=None):
        self.data     = data
        self.parent   = parent
        self.children = []

    @property
    def level(self):
        return self.data.get("level", 0)

    @property
    def title(self):
        return self.data.get("title", "")

    @property
    def path(self):
        return self.data.get("path", "")

    def __repr__(self):
        return f"TreeNode(level={self.level}, path={self.path!r}, title={self.title!r})"


def build_tree_nodes(tree_dict: dict, parent=None) -> TreeNode:
    """Recursively build TreeNode objects from the JSON tree dict."""
    node = TreeNode(tree_dict, parent)
    for child_dict in tree_dict.get("children", []):
        child_node = build_tree_nodes(child_dict, parent=node)
        node.children.append(child_node)
    return node


def find_section_for_page(root: TreeNode, page_number: int):
    """Return the section node containing the given page number."""
    for chapter in root.children:
        for section in chapter.children:
            page_ids = section.data.get("pageIds", [])
            # Look up page numbers for section page ids
            start = section.data.get("pageStart", 0)
            end   = section.data.get("pageEnd",   0)
            if start <= page_number <= end:
                return section
    return None


def find_chapter_for_page(root: TreeNode, page_number: int):
    """Return the chapter node containing the given page number."""
    for chapter in root.children:
        start = chapter.data.get("pageStart", 0)
        end   = chapter.data.get("pageEnd",   0)
        if start <= page_number <= end:
            return chapter
    return None


def get_path_to_page(root: TreeNode, page_number: int) -> list:
    """Return [root_node, chapter_node, section_node] for a page number."""
    chapter = find_chapter_for_page(root, page_number)
    if chapter is None:
        return [root]
    section = find_section_for_page(root, page_number)
    return [root, chapter, section] if section else [root, chapter]


def find_section_by_path(root: TreeNode, path: str):
    """Return a node by its path string, e.g. '2.3'."""
    for chapter in root.children:
        if chapter.path == path:
            return chapter
        for section in chapter.children:
            if section.path == path:
                return section
    return None


# Build in-memory tree from enriched_tree
tree_root = build_tree_nodes(enriched_tree)

print("Root :", tree_root)
print(f"Chapters: {len(tree_root.children)}")
for ch in tree_root.children:
    print(f"  [{ch.path}] {ch.title}  (pages {ch.data.get('pageStart')}–{ch.data.get('pageEnd')})")
    for sec in ch.children:
        print(f"    [{sec.path}] {sec.title}  ({sec.data.get('pageCount')} pages)")


Root : TreeNode(level=0, path='root', title='2023-annual-report-truncated')
Chapters: 5
  [1] Introduction  (pages 3–18)
    [1.1] Contents  (1 pages)
    [1.2] About the Federal Reserve  (1 pages)
    [1.3] Overview  (1 pages)
    [1.4] Additional Information  (1 pages)
    [1.5] Monetary Policy and Economic Developments  (10 pages)
  [2] Financial Stability  (pages 21–28)
    [2.1] Financial Stability Monitoring Framework  (2 pages)
    [2.2] Monitoring Financial Vulnerabilities  (2 pages)
    [2.3] Asset Valuation Pressures  (2 pages)
    [2.4] Borrowing by Households and Businesses  (2 pages)
  [3] Supervision and Regulation  (pages 29–34)
    [3.1] Supervision and Regulation  (2 pages)
    [3.2] Super vised and Regulated Institutions  (2 pages)
    [3.3] Bank Holding Companies  (2 pages)
  [4] Examinations and Inspections  (pages 35–38)
    [4.1] Examinations and Inspections  (2 pages)
    [4.2] Specialized Examinations  (2 pages)
  [5] Financial Stability and Regulatory Reports  

## 9. Demo — Resolve Page Number → Tree Path

In [108]:
test_page = pages[5]["pageNumber"] if len(pages) > 5 else pages[0]["pageNumber"]
path_nodes = get_path_to_page(tree_root, test_page)

print(f"Page {test_page} lives at:")
for node in path_nodes:
    indent = "  " * node.level
    ntype  = node.data.get("type", "root").upper()
    print(f"{indent}[{ntype}] {node.title}  (path={node.path})")


Page 6 lives at:
[ROOT] 2023-annual-report-truncated  (path=root)
  [CHAPTER] Introduction  (path=1)


## 10. Persist Tree to PostgreSQL
Upserts into the `Tree` table (unique on `documentId`).  
If a tree already exists for this document, `treeJson` is overwritten and `version` is bumped.

In [109]:
def upsert_tree(conn, document_id: str, tree_dict: dict):
    tree_json_str = json.dumps(tree_dict)

    with conn.cursor() as cur:
        cur.execute(
            'SELECT id, version FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        existing = cur.fetchone()

        if existing:
            tree_id, current_version = existing
            new_version = current_version + 1
            cur.execute(
                """
                UPDATE "Tree"
                SET
                    "treeJson"  = %s,
                    version     = %s,
                    "createdAt" = NOW()
                WHERE id = %s
                """,
                (tree_json_str, new_version, tree_id),
            )
            print(f"Updated existing tree (id={tree_id}) → version {new_version}")
        else:
            cur.execute(
                """
                INSERT INTO "Tree"
                    (id, "documentId", "treeJson", version)
                VALUES
                    (gen_random_uuid()::text, %s, %s, 1)
                """,
                (document_id, tree_json_str),
            )
            print(f"Inserted new tree for document {document_id}")

    conn.commit()
    print("Tree stored successfully.")


upsert_tree(conn, document_id, enriched_tree)


Inserted new tree for document DOC000001
Tree stored successfully.


## 11. Verify — Read Back from DB

In [110]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, "documentId", version, "treeJson"
        FROM "Tree"
        WHERE "documentId" = %s
        """,
        (document_id,),
    )
    row = cur.fetchone()

if row:
    tree_id, doc_id, version, stored_json = row
    if isinstance(stored_json, str):
        stored_json = json.loads(stored_json)

    print(f"tree.id      = {tree_id}")
    print(f"documentId   = {doc_id}")
    print(f"version      = {version}")
    print(f"root title   = {stored_json.get('title')}")
    print(f"chapters     = {len(stored_json.get('children', []))}")
    total_sections = sum(
        len(ch.get("children", []))
        for ch in stored_json.get("children", [])
    )
    print(f"sections     = {total_sections}")
    print(f"total pages  = {stored_json.get('pageCount')}")
else:
    print("No tree found for this document.")


tree.id      = 76602173-f929-45bd-89df-0df28878ed3b
documentId   = DOC000001
version      = 1
root title   = 2023-annual-report-truncated
chapters     = 5
sections     = 15
total pages  = 50


## 12. Reload Helper — Use in Other Notebooks
Copy `load_tree_from_db` and `TreeNode`/traversal helpers into `answerGeneration.ipynb`  
to load the tree and resolve which pages to fetch for a user query.

In [111]:
def load_tree_from_db(conn, document_id: str) -> TreeNode:
    """Load treeJson from DB and return the root TreeNode."""
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "treeJson" FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        row = cur.fetchone()

    if row is None:
        raise ValueError(f"No tree found for documentId={document_id!r}")

    tree_dict = row[0]
    if isinstance(tree_dict, str):
        tree_dict = json.loads(tree_dict)

    return build_tree_nodes(tree_dict)


# Demo reload
reloaded_root = load_tree_from_db(conn, document_id)
print("Reloaded:", reloaded_root)
for ch in reloaded_root.children:
    print(f"  [{ch.path}] {ch.title}  pages {ch.data['pageStart']}–{ch.data['pageEnd']}")
    for sec in ch.children:
        print(f"    [{sec.path}] {sec.title}  ({sec.data['pageCount']} pages)")


Reloaded: TreeNode(level=0, path='root', title='2023-annual-report-truncated')
  [1] Introduction  pages 3–18
    [1.1] Contents  (1 pages)
    [1.2] About the Federal Reserve  (1 pages)
    [1.3] Overview  (1 pages)
    [1.4] Additional Information  (1 pages)
    [1.5] Monetary Policy and Economic Developments  (10 pages)
  [2] Financial Stability  pages 21–28
    [2.1] Financial Stability Monitoring Framework  (2 pages)
    [2.2] Monitoring Financial Vulnerabilities  (2 pages)
    [2.3] Asset Valuation Pressures  (2 pages)
    [2.4] Borrowing by Households and Businesses  (2 pages)
  [3] Supervision and Regulation  pages 29–34
    [3.1] Supervision and Regulation  (2 pages)
    [3.2] Super vised and Regulated Institutions  (2 pages)
    [3.3] Bank Holding Companies  (2 pages)
  [4] Examinations and Inspections  pages 35–38
    [4.1] Examinations and Inspections  (2 pages)
    [4.2] Specialized Examinations  (2 pages)
  [5] Financial Stability and Regulatory Reports  pages 39–46
    [

## 13. Context Retrieval Helper — for Answer Generation
Given a page number (e.g. from a BM25 retrieval hit), returns the section context  
needed to fetch page content from the DB for answer generation.

In [112]:
def get_context_for_page(root: TreeNode, page_number: int) -> dict:
    """
    Given a page number, returns:
        root_title, chapter_title, section_title,
        page_ids (all pages in section), pageStart, pageEnd, path
    """
    path_nodes = get_path_to_page(root, page_number)

    chapter = next(
        (n for n in path_nodes if n.data.get("type") == "chapter"), None
    )
    section = next(
        (n for n in path_nodes if n.data.get("type") == "section"), None
    )

    return {
        "root_title":    root.title,
        "chapter_title": chapter.title if chapter else "",
        "chapter_path":  chapter.path  if chapter else "",
        "section_title": section.title if section else "",
        "section_path":  section.path  if section else "",
        "page_ids":      section.data.get("pageIds", []) if section else [],
        "pageStart":     section.data.get("pageStart")   if section else None,
        "pageEnd":       section.data.get("pageEnd")     if section else None,
    }


def fetch_section_content(conn, context: dict) -> list[dict]:
    """
    Given a context dict from get_context_for_page,
    fetches the actual page content rows from DB.
    Returns list of {pageNumber, content} dicts.
    """
    page_ids = context.get("page_ids", [])
    if not page_ids:
        return []

    placeholders = ",".join(["%s"] * len(page_ids))
    with conn.cursor() as cur:
        cur.execute(
            f'SELECT "pageNumber", content FROM "Page" '
            f'WHERE id IN ({placeholders}) ORDER BY "pageNumber"',
            page_ids,
        )
        rows = cur.fetchall()

    return [{"pageNumber": r[0], "content": r[1]} for r in rows]


# Demo
demo_page = pages[4]["pageNumber"] if len(pages) > 4 else pages[0]["pageNumber"]
ctx = get_context_for_page(tree_root, demo_page)
print("Context for page", demo_page)
print(json.dumps({k: v for k, v in ctx.items() if k != "page_ids"}, indent=2))
print(f"Section has {len(ctx['page_ids'])} pages")

section_pages = fetch_section_content(conn, ctx)
print(f"Fetched {len(section_pages)} page content rows")
if section_pages:
    print("First page snippet:", section_pages[0]["content"][:200])


Context for page 5
{
  "root_title": "2023-annual-report-truncated",
  "chapter_title": "Introduction",
  "chapter_path": "1",
  "section_title": "About the Federal Reserve",
  "section_path": "1.2",
  "pageStart": 5,
  "pageEnd": 5
}
Section has 1 pages
Fetched 1 page content rows
First page snippet: About the F ederal Reser ve
The F ederal Reser ve was created b y an act of Congress on December 23, 1913, to pro vide the
nation with a safer , more flexible, and more stable monetar y and financial 
